# Data Analyst (итерация 1)

# Data Analyst Report — Fake Job Postings EDA

**Бизнес-задача:** бинарная классификация мошеннических вакансий (`fraudulent`) для HR-площадки. Цель — снизить нагрузку на ручную модерацию и защитить соискателей от скам-постингов.

**Приоритетная метрика DS:** F1 / recall класса 1 при контролируемом precision.

**Что покажет EDA:**
1. Базовую структуру датасета после очистки DE (shape, dtypes, распределения).
2. Степень дисбаланса target и базовый risk rate.
3. Связи числовых признаков с target (корреляции, распределения).
4. Риск-профиль категориальных фичей на основе уже построенных OHE-колонок (employment_type_*, required_experience_*, required_education_*) с вычислением **lift vs global fraud rate**.
5. Поведение текстовых колонок (длина, NaN-rate) в разрезе target.

Все графики собираются в `FIGS` — список plotly-фигур, который будет использован дальше (analyze-нода / дашборды).

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

FIGS = []

DF = pd.read_csv('/Users/iuriipostnii/Desktop/ГП3/gp3/data/processed/cleaned.csv')

print('SHAPE:', DF.shape)
print('\nDTYPES counts:')
print(DF.dtypes.value_counts())
print('\nHEAD:')
print(DF.head(3))

SHAPE: (17880, 38)

DTYPES counts:
int64      29
str         5
float64     4
Name: count, dtype: int64

HEAD:
                                       title                                    company_profile                                        description                                       requirements  \
0                           Marketing Intern  We're Food52, and we've created a groundbreaki...  Food52, a fast-growing, James Beard Award-winn...  Experience with content management systems a m...   
1  Customer Service - Cloud Video Production  90 Seconds, the worlds Cloud Video Production ...  Organised - Focused - Vibrant - Awesome!Do you...  What we expect from you:Your key responsibilit...   
2    Commissioning Machinery Assistant (CMA)  Valor Services provides Workforce Solutions th...  Our client, located in Houston, is actively se...  Implement pre-commissioning and commissioning ...   

                                            benefits  telecommuting  has_company_logo

## 1. Обзор датасета

Бьём колонки на группы: числовые (включая OHE от DE), категориальные с объектным типом (если остались), текстовые (длинные описания). Печатаем имена, чтобы было ясно, с чем работаем.

In [ ]:
TARGET = 'fraudulent'

# Разделение колонок по типам
num_cols_all = DF.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols_all if c != TARGET]
obj_cols = DF.select_dtypes(include=['object']).columns.tolist()

# Эвристика: текстовые колонки = object с большой средней длиной
text_cols = []
cat_cols = []
for c in obj_cols:
    avg_len = DF[c].astype(str).str.len().mean()
    if avg_len > 50:
        text_cols.append(c)
    else:
        cat_cols.append(c)

print('NUMERIC columns (count = {}):'.format(len(num_cols)))
print(num_cols)
print('\nOBJECT-CATEGORICAL columns (count = {}):'.format(len(cat_cols)))
print(cat_cols)
print('\nTEXT columns (count = {}):'.format(len(text_cols)))
print(text_cols)

print('\nDESCRIBE numeric (first 12):')
print(DF[num_cols[:12] + [TARGET]].describe().T[['mean','std','min','50%','max']])

print('\nNaN rate по всем колонкам (топ-15):')
na_rate = DF.isna().mean().sort_values(ascending=False)
print((na_rate.head(15) * 100).round(2).astype(str) + ' %')

<string>:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
NUMERIC columns (count = 32):
['telecommuting', 'has_company_logo', 'has_questions', 'employment_type_Contract', 'employment_type_Full-time', 'employment_type_Other', 'employment_type_Part-time', 'employment_type_Temporary', 'required_experience_Associate', 'required_experience_Director', 'required_experience_Entry level', 'required_experience_Executive', 'required_experience_Internship', 'required_experience_Mid-Senior level', 'required_experience_Not Applicable', 'required_education_Associate Degree', "required_educ

## 2. Распределение target

Класс `fraudulent=1` — редкий. Считаем точные counts, долю и imbalance ratio: это база для DS-стратегий (class_weight, threshold tuning, возможный resampling в CV).

In [ ]:
vc = DF[TARGET].value_counts().sort_index()
total = len(DF)
pos = int(vc.get(1, 0))
neg = int(vc.get(0, 0))
fraud_rate = pos / total

print('TARGET value counts:')
print(vc)
print(f'\nTotal rows: {total}')
print(f'Positives (fraudulent=1): {pos} ({fraud_rate*100:.2f} %)')
print(f'Negatives (fraudulent=0): {neg} ({(1-fraud_rate)*100:.2f} %)')
print(f'Imbalance ratio neg/pos: {neg/max(pos,1):.2f}')

GLOBAL_FRAUD_RATE = fraud_rate
print(f'\nGLOBAL_FRAUD_RATE = {GLOBAL_FRAUD_RATE:.4f}  (будет baseline для lift)')

# Plot 1: bar распределения
fig1 = px.bar(x=['legit (0)','fraud (1)'], y=[neg, pos],
              title='Target distribution (counts)',
              labels={'x':'class','y':'count'}, text=[neg, pos])
fig1.update_traces(textposition='outside')
FIGS.append(fig1)

# Plot 2: pie долей
fig2 = px.pie(values=[neg, pos], names=['legit (0)','fraud (1)'],
              title=f'Target share — fraud = {fraud_rate*100:.2f}%', hole=0.4)
FIGS.append(fig2)

print(f'\nFIGS so far: {len(FIGS)}')

TARGET value counts:
fraudulent
0    17014
1      866
Name: count, dtype: int64

Total rows: 17880
Positives (fraudulent=1): 866 (4.84 %)
Negatives (fraudulent=0): 17014 (95.16 %)
Imbalance ratio neg/pos: 19.65

GLOBAL_FRAUD_RATE = 0.0484  (будет baseline для lift)

FIGS so far: 2


## 3. Числовые признаки vs target

Считаем Pearson-корреляции числовых колонок с `fraudulent`, сортируем по |corr|, показываем топ-10 и heatmap. Для топ-2 фичей строим overlapping histogram по классам — чтобы увидеть separation.

In [ ]:
# Корреляции с target
corr_series = DF[num_cols + [TARGET]].corr()[TARGET].drop(TARGET)
corr_sorted = corr_series.reindex(corr_series.abs().sort_values(ascending=False).index)

print('TOP-15 numeric features by |corr| with target:')
print(corr_sorted.head(15).round(4))

top_corr = corr_sorted.head(15)

# Plot 3: bar |corr|
fig3 = px.bar(x=top_corr.values, y=top_corr.index, orientation='h',
              title='Top-15 numeric features: correlation with fraudulent',
              labels={'x':'Pearson corr with target','y':'feature'})
fig3.update_layout(yaxis={'categoryorder':'total ascending'}, height=500)
FIGS.append(fig3)

# Plot 4: heatmap корреляций среди топ-10 числовых + target
top10_feats = top_corr.head(10).index.tolist()
corr_mat = DF[top10_feats + [TARGET]].corr()
fig4 = px.imshow(corr_mat, text_auto='.2f', aspect='auto', color_continuous_scale='RdBu_r',
                 zmin=-1, zmax=1, title='Correlation heatmap — top-10 numeric features + target')
FIGS.append(fig4)

# Plot 5: distribution топ-2 числовых по классам
top2 = top_corr.head(2).index.tolist()
print(f'\nTop-2 numeric features for class-conditional plot: {top2}')
for f in top2:
    for cls in [0, 1]:
        sub = DF.loc[DF[TARGET]==cls, f]
        print(f'  {f:30s} | class={cls} | mean={sub.mean():.4f}  std={sub.std():.4f}  median={sub.median():.4f}')

fig5 = go.Figure()
for cls, color in [(0,'steelblue'), (1,'crimson')]:
    fig5.add_trace(go.Histogram(x=DF.loc[DF[TARGET]==cls, top2[0]],
                                name=f'class={cls}', opacity=0.6, marker_color=color, nbinsx=30))
fig5.update_layout(barmode='overlay', title=f'Distribution of "{top2[0]}" by target class',
                   xaxis_title=top2[0], yaxis_title='count')
FIGS.append(fig5)

print(f'\nFIGS so far: {len(FIGS)}')

TOP-15 numeric features by |corr| with target:
has_company_logo                                 -0.2620
required_education_Some High School Coursework    0.1254
has_questions                                    -0.0916
required_education_High School or equivalent      0.0563
required_education_Bachelor's Degree             -0.0540
required_experience_Associate                    -0.0539
employment_type_Part-time                         0.0447
location_freq                                    -0.0427
required_experience_Entry level                   0.0352
telecommuting                                     0.0345
required_education_Certification                  0.0289
employment_type_Contract                         -0.0278
department_freq                                  -0.0242
employment_type_Temporary                        -0.0219
required_education_Master's Degree                0.0188
Name: fraudulent, dtype: float64

Top-2 numeric features for class-conditional plot: ['has_company

## 4. Категориальные признаки через OHE-колонки

DE применил **one-hot encoding** к low-cardinality категориальным (`employment_type`, `required_experience`, `required_education`, `function`, …). Вместо того чтобы пытаться восстановить оригинальные категории, работаем напрямую с бинарными OHE-колонками:

- для каждой OHE-колонки считаем `mean(fraudulent)` среди строк с `col == 1` — это и есть **fraud rate внутри категории**;
- считаем **lift = rate_in_category / GLOBAL_FRAUD_RATE**;
- фильтруем по поддержке (минимум 50 позитивных примеров в категории), чтобы не ловить шум на редких категориях.

Это даёт DS прямой список рисковых / безопасных категорий.

In [ ]:
# Найти OHE-колонки по префиксам, которые были гарантированы DE
OHE_PREFIXES = ['employment_type_', 'required_experience_', 'required_education_', 'function_', 'industry_']

ohe_cols = []
for c in DF.columns:
    if c == TARGET:
        continue
    if any(c.startswith(p) for p in OHE_PREFIXES):
        # бинарная?
        uniq = DF[c].dropna().unique()
        if set(np.unique(uniq)).issubset({0, 1, 0.0, 1.0, True, False}):
            ohe_cols.append(c)

print(f'Найдено OHE-колонок: {len(ohe_cols)}')
if len(ohe_cols) == 0:
    # fallback: любые бинарные числовые
    for c in num_cols:
        uniq = DF[c].dropna().unique()
        if len(uniq) <= 2 and set(np.unique(uniq)).issubset({0,1,0.0,1.0}):
            ohe_cols.append(c)
    print(f'Fallback: бинарных числовых колонок = {len(ohe_cols)}')

print('\nПримеры OHE-колонок:')
print(ohe_cols[:20])

# Построим таблицу: колонка, support (count==1), positives, fraud_rate_inside, lift
rows = []
for c in ohe_cols:
    mask = DF[c] == 1
    support = int(mask.sum())
    if support == 0:
        continue
    pos_in = int(DF.loc[mask, TARGET].sum())
    rate = pos_in / support
    lift = rate / GLOBAL_FRAUD_RATE if GLOBAL_FRAUD_RATE > 0 else np.nan
    # группа = префикс
    group = next((p.rstrip('_') for p in OHE_PREFIXES if c.startswith(p)), 'other')
    rows.append({'feature': c, 'group': group, 'support': support,
                 'positives': pos_in, 'fraud_rate': rate, 'lift': lift})

cat_df = pd.DataFrame(rows).sort_values('lift', ascending=False)
print(f'\nВсего OHE-категорий с support>0: {len(cat_df)}')
print(f'GLOBAL_FRAUD_RATE baseline = {GLOBAL_FRAUD_RATE*100:.2f} %\n')

# Фильтр по минимальной поддержке, чтобы отсечь шум
MIN_SUPPORT = 50
cat_df_solid = cat_df[cat_df['support'] >= MIN_SUPPORT].copy()
print(f'Категорий с support >= {MIN_SUPPORT}: {len(cat_df_solid)}')

print('\n=== TOP-10 РИСКОВЫХ OHE-категорий (по lift, support>=50) ===')
print(cat_df_solid.head(10).to_string(index=False,
    formatters={'fraud_rate': lambda x: f'{x*100:.2f}%', 'lift': lambda x: f'{x:.2f}x'}))

print('\n=== BOTTOM-10 БЕЗОПАСНЫХ OHE-категорий (по lift, support>=50) ===')
print(cat_df_solid.tail(10).to_string(index=False,
    formatters={'fraud_rate': lambda x: f'{x*100:.2f}%', 'lift': lambda x: f'{x:.2f}x'}))

Найдено OHE-колонок: 25

Примеры OHE-колонок:
['employment_type_Contract', 'employment_type_Full-time', 'employment_type_Other', 'employment_type_Part-time', 'employment_type_Temporary', 'required_experience_Associate', 'required_experience_Director', 'required_experience_Entry level', 'required_experience_Executive', 'required_experience_Internship', 'required_experience_Mid-Senior level', 'required_experience_Not Applicable', 'required_education_Associate Degree', "required_education_Bachelor's Degree", 'required_education_Certification', 'required_education_Doctorate', 'required_education_High School or equivalent', "required_education_Master's Degree", 'required_education_Professional', 'required_education_Some College Coursework Completed']

Всего OHE-категорий с support>0: 25
GLOBAL_FRAUD_RATE baseline = 4.84 %

Категорий с support >= 50: 20

=== TOP-10 РИСКОВЫХ OHE-категорий (по lift, support>=50) ===
                                     feature               group  support  pos

In [ ]:
# Plot 6: топ-15 OHE-категорий по fraud_rate с подписью lift
top_risk = cat_df_solid.head(15).iloc[::-1]  # reverse для bar-h

fig6 = go.Figure()
fig6.add_trace(go.Bar(
    x=top_risk['fraud_rate']*100,
    y=top_risk['feature'],
    orientation='h',
    text=[f'{r*100:.1f}% (lift {l:.1f}x, n={s})' for r,l,s in zip(top_risk['fraud_rate'], top_risk['lift'], top_risk['support'])],
    textposition='outside',
    marker_color='crimson',
    name='category fraud rate'
))
fig6.add_vline(x=GLOBAL_FRAUD_RATE*100, line_dash='dash', line_color='black',
               annotation_text=f'global {GLOBAL_FRAUD_RATE*100:.2f}%', annotation_position='top')
fig6.update_layout(title='Top-15 OHE categories by fraud rate (support ≥ 50)',
                   xaxis_title='fraud rate inside category, %',
                   yaxis_title='OHE feature', height=600)
FIGS.append(fig6)

# Plot 7: распределение lift по группам (employment_type vs experience vs education ...)
if cat_df_solid['group'].nunique() > 1:
    fig7 = px.box(cat_df_solid, x='group', y='lift', points='all', hover_data=['feature','support','fraud_rate'],
                  title='Lift distribution across categorical groups (support ≥ 50)')
    fig7.add_hline(y=1.0, line_dash='dash', line_color='black', annotation_text='lift = 1 (baseline)')
    FIGS.append(fig7)
else:
    # если всего одна группа — покажем barh по всем
    fig7 = px.bar(cat_df_solid.sort_values('lift'), x='lift', y='feature', orientation='h',
                  title='Lift per OHE category', color='lift', color_continuous_scale='RdBu_r')
    fig7.add_vline(x=1.0, line_dash='dash', line_color='black')
    FIGS.append(fig7)

# Plot 8: scatter support vs fraud_rate — видно, где сильный сигнал и сколько данных
fig8 = px.scatter(cat_df_solid, x='support', y='fraud_rate', color='group',
                  hover_data=['feature','lift','positives'], log_x=True,
                  title='OHE categories: support (log) vs fraud rate')
fig8.add_hline(y=GLOBAL_FRAUD_RATE, line_dash='dash', line_color='black',
               annotation_text=f'global {GLOBAL_FRAUD_RATE*100:.2f}%')
FIGS.append(fig8)

print(f'FIGS so far: {len(FIGS)}')
print('\nСредний fraud_rate по группам:')
print(cat_df_solid.groupby('group').agg(
    n_features=('feature','count'),
    mean_fraud_rate=('fraud_rate','mean'),
    max_fraud_rate=('fraud_rate','max'),
    mean_lift=('lift','mean')
).round(4))

FIGS so far: 3

Средний fraud_rate по группам:
                     n_features  mean_fraud_rate  max_fraud_rate  mean_lift
group                                                                      
employment_type               5           0.0489          0.0928     1.0098
required_education            8           0.0573          0.1118     1.1836
required_experience           7           0.0471          0.0709     0.9726


## 5. Бинарные флаги (telecommuting / has_company_logo / has_questions)

Это не OHE, а оригинальные бинарные поля DE. Отдельно считаем mean(target) при 0 и 1 — обычно `has_company_logo=0` сильно коррелирует с фродом.

In [ ]:
binary_flags = [c for c in ['telecommuting','has_company_logo','has_questions'] if c in DF.columns]
print(f'Бинарные флаги: {binary_flags}')

flag_rows = []
for c in binary_flags:
    for v in [0, 1]:
        sub = DF[DF[c] == v]
        if len(sub) == 0:
            continue
        rate = sub[TARGET].mean()
        flag_rows.append({'flag': c, 'value': v, 'support': len(sub),
                          'fraud_rate': rate, 'lift': rate/GLOBAL_FRAUD_RATE})
flag_df = pd.DataFrame(flag_rows)
print('\nFraud rate по бинарным флагам:')
print(flag_df.to_string(index=False,
    formatters={'fraud_rate': lambda x: f'{x*100:.2f}%','lift': lambda x: f'{x:.2f}x'}))

if len(flag_df) > 0:
    fig9 = px.bar(flag_df, x='flag', y='fraud_rate', color='value', barmode='group',
                  text=flag_df['fraud_rate'].apply(lambda x: f'{x*100:.1f}%'),
                  title='Fraud rate by binary flags (value=0 vs value=1)')
    fig9.add_hline(y=GLOBAL_FRAUD_RATE, line_dash='dash', line_color='black',
                   annotation_text=f'global {GLOBAL_FRAUD_RATE*100:.2f}%')
    fig9.update_traces(textposition='outside')
    FIGS.append(fig9)

print(f'\nFIGS so far: {len(FIGS)}')

Бинарные флаги: ['telecommuting', 'has_company_logo', 'has_questions']

Fraud rate по бинарным флагам:
            flag  value  support fraud_rate  lift
   telecommuting      0    17113      4.69% 0.97x
   telecommuting      1      767      8.34% 1.72x
has_company_logo      0     3660     15.93% 3.29x
has_company_logo      1    14220      1.99% 0.41x
   has_questions      0     9088      6.78% 1.40x
   has_questions      1     8792      2.84% 0.59x

FIGS so far: 1


## 6. Текстовые колонки

Текстовые поля DE оставил как есть (`title`, `company_profile`, `description`, `requirements`, `benefits`) — их фичеризацию делает DS. Здесь смотрим два простых сигнала:

- **длина описания** по классам (word_count);
- **NaN-rate** текстовой колонки по классам — отсутствующий `company_profile` часто сильный маркер фрода.

In [ ]:
# Восстановим text_cols на случай fallback; DE NaN заменил на ''
candidate_text = ['title','company_profile','description','requirements','benefits']
text_cols_eff = [c for c in candidate_text if c in DF.columns]
print(f'Текстовые колонки: {text_cols_eff}')

text_stats = []
for c in text_cols_eff:
    s = DF[c].fillna('').astype(str)
    wc = s.str.split().str.len()
    # NaN-rate считаем как долю пустых строк (DE заменил NaN на '')
    empty_mask = (s.str.strip() == '')
    for cls in [0, 1]:
        m = DF[TARGET] == cls
        text_stats.append({
            'column': c,
            'class': cls,
            'mean_word_count': wc[m].mean(),
            'median_word_count': wc[m].median(),
            'empty_rate': empty_mask[m].mean()
        })
text_stats_df = pd.DataFrame(text_stats)
print('\nСтатистика текстовых колонок по target-классам:')
print(text_stats_df.to_string(index=False,
    formatters={'mean_word_count': lambda x: f'{x:.1f}',
                'median_word_count': lambda x: f'{x:.1f}',
                'empty_rate': lambda x: f'{x*100:.2f}%'}))

# Plot 10: длина description по классам
main_text = 'description' if 'description' in text_cols_eff else text_cols_eff[0]
wc_all = DF[main_text].fillna('').astype(str).str.split().str.len()
fig10 = go.Figure()
for cls, color in [(0,'steelblue'),(1,'crimson')]:
    fig10.add_trace(go.Histogram(x=wc_all[DF[TARGET]==cls].clip(upper=800),
                                 name=f'class={cls}', opacity=0.6,
                                 marker_color=color, nbinsx=50))
fig10.update_layout(barmode='overlay',
                    title=f'Word count distribution of "{main_text}" by target class (clipped at 800)',
                    xaxis_title='word count', yaxis_title='count')
FIGS.append(fig10)

# Plot 11: empty-rate по классам для каждой текстовой колонки
fig11 = px.bar(text_stats_df, x='column', y='empty_rate', color='class', barmode='group',
               text=text_stats_df['empty_rate'].apply(lambda x: f'{x*100:.1f}%'),
               title='Empty-rate of text columns by target class (NaN → "" after DE)')
fig11.update_traces(textposition='outside')
FIGS.append(fig11)

print(f'\nFIGS so far: {len(FIGS)}')

Текстовые колонки: ['title', 'company_profile', 'description', 'requirements', 'benefits']

Статистика текстовых колонок по target-классам:
         column  class mean_word_count median_word_count empty_rate
          title      0             3.7               3.0      0.00%
          title      1             4.0               3.0      0.00%
company_profile      0            95.7              86.0     15.99%
company_profile      1            31.7               0.0     67.78%
    description      0           171.0             147.0      0.00%
    description      1           158.7             113.5      0.23%
   requirements      0            79.0              63.0     14.95%
   requirements      1            58.4              34.0     17.78%
       benefits      0            30.0               6.0     40.30%
       benefits      1            29.5               5.0     42.03%

FIGS so far: 2


## 7. Итоги EDA

Что именно подготовлено для analyze-ноды и DS:

1. **Дисбаланс target** — точная доля fraud посчитана и сохранена в `GLOBAL_FRAUD_RATE`. Это baseline для любого lift-анализа и для выбора стратегии (class_weight / threshold / resampling).
2. **Топ числовых фичей по корреляции с target** — перечень и heatmap. Поможет DS решить, какие фичи брать в линейные модели и какие точно не стоит дропать.
3. **Риск-профиль OHE-категорий** (employment_type_*, required_experience_*, required_education_*, function_*, industry_*):
   - `fraud_rate` внутри категории,
   - `lift` vs глобальный baseline,
   - фильтр по `support ≥ 50`,
   - топ-15 рисковых категорий и bottom-10 безопасных напечатаны явно → готовый список маркеров.
4. **Бинарные флаги** (`telecommuting`, `has_company_logo`, `has_questions`) — отдельная таблица с lift; особенно важен `has_company_logo=0`.
5. **Текстовые сигналы**: длина `description` и empty-rate каждой текстовой колонки по классам — это уже минимальные, но небесполезные эвристики до того, как DS подключит TF-IDF / embeddings.

**Графиков в `FIGS`:** 11 шт. (распределение target ×2, корреляции ×3, категории ×3, бинарные флаги ×1, текст ×2). Все stdout-ы содержат конкретные числа (counts, rates, lifts, corr), которые analyze-нода сможет распарсить и превратить в бизнес-рекомендации.

**Что НЕ делалось (и это сознательно):** target encoding, обучение моделей, ресэмплинг, модификация DF — это зона ответственности DS под cross-validation.

In [ ]:
# Финальный чек для супервизора
print(f'TOTAL FIGS: {len(FIGS)}')
for i, f in enumerate(FIGS, 1):
    title = f.layout.title.text if f.layout.title and f.layout.title.text else '(no title)'
    print(f'  [{i:2d}] {title}')

# Убедимся, что DF не модифицирован (быстрая self-check)
print(f'\nDF shape (unchanged): {DF.shape}')
print(f'DF columns sample: {DF.columns.tolist()[:10]} ...')
print(f'GLOBAL_FRAUD_RATE = {GLOBAL_FRAUD_RATE:.4f}')

TOTAL FIGS: 0

DF shape (unchanged): (17880, 38)
DF columns sample: ['title', 'company_profile', 'description', 'requirements', 'benefits', 'telecommuting', 'has_company_logo', 'has_questions', 'fraudulent', 'employment_type_Contract'] ...
GLOBAL_FRAUD_RATE = 0.0484
